# S5.M5-1 Orange 大版三任务 + M3 两方特征预筛

**目的**：方法筛选第三阶段（第二批）第 1 步（[DR-M5-014](../../../registry/decision_records/DR-M5-014.yaml) · D-1 ~ D-3）。在 Orange KDD Cup 2009 大版（15,000 个变量）的流失 / 追加销售 / 偏好三个任务上：
- 先做 M3 两方特征预筛，再走 S5.M1 的全部方法与 S5.M3 的交叉验证选组合，看 14 个任务上的结论在高维数据上是否仍成立；
- 检验预筛本身是否损失精度：每方保留 300 列相对全部清洗后列，配对差区间下界 > −0.005 判「不损失」（D-3，事先写定）。

**做法**：
- 测试集、验证集、两方划分与小版同口径（分层 20% / 20%，原始变量随机对半）；大版官方测试集不完整，测试集在训练集内构造；
- 删列、缺失指示与序数只在 tune 视图 fit 行（训练去掉验证）上判定；
- 内地按本方 XGBoost 总增益、香港按自己特征在联邦预拟合中被选为分裂的次数，各取前 300 列；
- SGB 中被动方本来就保存自己特征上的分裂，这一步不让主动方多知道香港列的身份。

**数据**：公开数据，**开发证据**，不代表汇丰效果。矩阵与逐行打分只在私有目录。

In [1]:
import hashlib
import json
import os
import pathlib
import subprocess
import sys
import tempfile

import numpy as np
import pandas as pd
import yaml

HASH_PREFIX_LEN = 12
REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "AGENTS.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
CONFIG_PATH = REPO_ROOT / "modules/m5_modeling/configs/s5_m5_orange_large.yaml"
m5 = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
cfg = yaml.safe_load((REPO_ROOT / m5["base_config"]).read_text(encoding="utf-8"))
m3 = yaml.safe_load((REPO_ROOT / m5["cv_config"]).read_text(encoding="utf-8"))
COMPONENTS = [REPO_ROOT / "modules/m5_modeling/components" / n
              for n in ("method_screen.py", "method_screen_data.py", "method_screen_cv.py", "method_screen_large.py")]
TRACKED = [CONFIG_PATH, REPO_ROOT / m5["base_config"], REPO_ROOT / m5["cv_config"], REPO_ROOT / m5["decision_record"]]


def sha(path, n=HASH_PREFIX_LEN):
    return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()[:n]


git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True, cwd=REPO_ROOT).strip()
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                       cwd=REPO_ROOT).stdout.strip() != ""
for p in TRACKED + COMPONENTS:
    print(p.relative_to(REPO_ROOT), "sha256:" + sha(p))
print("决策记录:", m5["decision_record"], "| step:", m5["step_id"], "|", m5["purpose"])
print("seed:", m5["seed"], "| seeds:", m5["seeds"], "| k_main:", m5["k_main"], "| k_sens:", m5["k_sens"])
print("git:", git_sha + ("（工作区含本步未提交改动）" if dirty else ""))
PRIV = pathlib.Path(os.environ[cfg["private_root_env"]])
RAW = pathlib.Path(os.environ[cfg["raw_root_env"]])
RUN = pathlib.Path(os.environ[m5["run_dir_env"]])
launch = {}
print("启动指纹（运行开始时记录；路径只留文件名）：")
for line in (RUN / "fingerprint_launch.txt").read_text(encoding="utf-8").splitlines():
    parts = line.split()
    if len(parts) == 2 and len(parts[0]) == hashlib.sha256().digest_size * 2:
        launch[pathlib.Path(parts[1]).name] = parts[0]
        print("  sha256:" + parts[0][:HASH_PREFIX_LEN], pathlib.Path(parts[1]).name)
    else:
        print(" ", line)
for p in COMPONENTS + [CONFIG_PATH]:
    assert launch[p.name] == hashlib.sha256(p.read_bytes()).hexdigest(), p.name
print("运行时组件与配置与当前仓库逐字节相同。")

modules/m5_modeling/configs/s5_m5_orange_large.yaml sha256:fd06cd6811df
modules/m5_modeling/configs/s5_m1_method_screen.yaml sha256:761bae66da90
modules/m5_modeling/configs/s5_m3_cv_combo.yaml sha256:6841f6884602
registry/decision_records/DR-M5-014.yaml sha256:1e465cbb281d
modules/m5_modeling/components/method_screen.py sha256:05e19e025d0b
modules/m5_modeling/components/method_screen_data.py sha256:e4bfdf6fe3d4
modules/m5_modeling/components/method_screen_cv.py sha256:5f7a6b1b108c
modules/m5_modeling/components/method_screen_large.py sha256:5121a5b9368c
决策记录: registry/decision_records/DR-M5-014.yaml | step: S5.M5 | Orange 大版（15,000 列）三任务上，M3 两方特征预筛之后，S5.M1 的方法与 S5.M3 的组合选择是否仍成立；预筛本身是否损失精度
seed: 11 | seeds: [11, 22, 33, 44, 55, 66] | k_main: 300 | k_sens: [100, 300, 1000, 'all']
git: 5aeb7e2（工作区含本步未提交改动）
启动指纹（运行开始时记录；路径只留文件名）：
  launch 2026-11-03T21:04:36+08:00 git 674a299
  sha256:1e465cbb281d DR-M5-014.yaml
  sha256:2dede26c26f6 method_screen_large.py
  sha256:fd06cd6811df s5_m5_orang

## 1. 三个任务的规模与预筛

列数为编码后的列（数值列 + 缺失指示；内地类别列为目标编码与频次两列，香港类别列为频次与序数）。「非零」= 预筛模型中增益或分裂次数大于 0 的列。

In [2]:
from modules.m5_modeling.components import method_screen as MS
from modules.m5_modeling.components import method_screen_cv as CV
from modules.m5_modeling.components import method_screen_data as MD
from modules.m5_modeling.components import method_screen_large as ML

P = MS.Params(cfg)
NAMES = [ML.name_of(lb) for lb in m5["labels"]]
meta = {n: json.loads((PRIV / (n + ".json")).read_text(encoding="utf-8")) for n in NAMES}
rows = []
for n in NAMES:
    m, pre = meta[n], meta[n]["prescreen"]
    rows.append({"task": n, "fit_rows": m["final_fit_rows"], "fit_pos": m["final_fit_pos"], "test_rows": m["final_eval_rows"],
                 "test_pos": m["final_eval_pos"], "vars_guest": pre["vars_guest"], "vars_host": pre["vars_host"],
                 "clean_guest": pre["clean_guest"], "clean_host": pre["clean_host"],
                 "nonzero_guest": pre["nonzero_guest"], "nonzero_host": pre["nonzero_host"],
                 "kept_guest": m["n_guest"], "kept_host": m["n_host"]})
tasks = pd.DataFrame(rows)
print("三个任务的规模与 M3 预筛（列数为编码后的列）：")
print(tasks.to_string(index=False))

三个任务的规模与 M3 预筛（列数为编码后的列）：


                  task  fit_rows  fit_pos  test_rows  test_pos  vars_guest  vars_host  clean_guest  clean_host  nonzero_guest  nonzero_host  kept_guest  kept_host
    orange_large_churn     40000     2938      10000       734        7500       7500         6655        6640           1088           872         300        300
   orange_large_upsell     40000     2946      10000       736        7500       7500         6662        6646            967           710         300        300
orange_large_appetency     40000      712      10000       178        7500       7500         6655        6646            913           701         300        300


## 2. 复现

下一格从原始分块重做流失任务的预筛，核对两方列清单与批量运行相同；再用私有矩阵与批量运行选定的配置重算优化版联邦的测试 AUROC。

In [3]:
TASK = m5["repro_task"]
SRC = RAW / m5["raw_subdir"]
TMP = pathlib.Path(tempfile.mkdtemp())
kept = ML.scan_numeric(SRC)
num, cat = ML.read_raw(SRC, num_vars=kept, all_num=ML.LARGE_NUM, num_path=TMP / "num.npy")
print("从原始分块读取：%d / %d 个数值变量非空且非常数，%d 行" % (len(kept), len(ML.LARGE_NUM), num.shape[0]))
y = ML.read_labels(SRC, TASK)
te, va = MD.simple_split(y, cfg["prep"])
plan, rank_g, rank_h = ML.select_columns(num, cat, y, ~te & ~va, ML.party_vars(int(cfg["prep"]["split_seed"])), P,
                                          int(m5["k_main"]), kept)
gn, hn = ML.encoded_names(plan, "g"), ML.encoded_names(plan, "h")
same_g = [gn[j] for j in sorted(rank_g[:m5["k_main"]])] == meta[ML.name_of(TASK)]["guest_cols"]
same_h = [hn[j] for j in sorted(rank_h[:m5["k_main"]])] == meta[ML.name_of(TASK)]["host_cols"]
del num, cat
(TMP / "num.npy").unlink()
sc = MS.Screen(ML.name_of(TASK), cfg, PRIV, RUN / "unused")
res = json.loads((RUN / (ML.name_of(TASK) + ".json")).read_text(encoding="utf-8"))
fv = sc.v["final"]
pf = MS.prior_fn(P)(fv["fit_y"])
cc = res["configs"]["central"]
redo = [MS.auc(fv["ev_y"], MS.train_pred(P, fv["fit_xc"], fv["fit_y"], fv["ev_xc"], cc, P.bins_fine, pf, sd))
        for sd in ([P.seeds[0]] if MS.deterministic(cc) else P.seeds)]
print("从原始分块重做 %s 的预筛：内地列清单相同 %s，香港列清单相同 %s" % (TASK, same_g, same_h))
print("优化版联邦测试 AUROC：重算 %.6f，批量运行 %.6f" % (np.mean(redo), res["test"]["F_opt_p"]["auroc"]))
assert same_g and same_h and abs(np.mean(redo) - res["test"]["F_opt_p"]["auroc"]) < m5["repro_tol"]

从原始分块读取：13261 / 14740 个数值变量非空且非常数，50000 行


从原始分块重做 churn 的预筛：内地列清单相同 True，香港列清单相同 True
优化版联邦测试 AUROC：重算 0.754471，批量运行 0.754471


## 3. 结果

- 方法对比与 S5.M1 同口径：测试集 6 个 seed，两层 bootstrap（测试行 × seed，1,000 次），数据集内 Holm；
- 组合选择同时列 S5.M1 单一验证集与 S5.M3 交叉验证（分层 5 折）的 R1 / R2；
- 敏感性在 final 视图上用主分析选定的配置，k = 100 / 300 / 1000 / 全部。

In [4]:
OUT = REPO_ROOT / m5["results_dir"]
OUT.mkdir(parents=True, exist_ok=True)
KEEP = ["F_opt_p − F_def", "F_def − G_t", "F_opt_p − G_t", "F_M6 − F_opt_p", "F_M1 − F_opt_p", "F_M4 − F_opt_p",
        "COMBO − F_opt_p"]
con, cvc, cvt, sens = [], [], [], []
for n in NAMES:
    r = json.loads((RUN / (n + ".json")).read_text(encoding="utf-8"))
    for c in r["contrasts"]:
        lab = "%s − %s" % (c["a"], c["b"])
        if lab in KEEP:
            con.append({"task": n, "contrast": lab, "diff": c["diff"], "ci_low": c["ci_low"], "ci_high": c["ci_high"],
                        "p": c["p"], "p_holm": c.get("p_holm"), "a_auroc": r["test"][c["a"]]["auroc"]})
    v = json.loads((RUN / (n + "__cv.json")).read_text(encoding="utf-8"))
    cvc.append({"task": n, "s5m1_choice": v["s5m1"]["choice"], "cv_r1": v["choice"]["r1"], "cv_r2": v["choice"]["r2"],
                "P_test": v["test"]["P"]["auroc"], "R1_test": v["test"]["CV_R1"]["auroc"], "R2_test": v["test"]["CV_R2"]["auroc"]})
    for c in v["contrasts"]:
        cvt.append({"task": n, "a": c["a"], "b": c["b"], "a_pick": v["choice"]["r1" if c["a"] == "CV_R1" else "r2"],
                    "diff": c["diff"], "ci_low": c["ci_low"], "ci_high": c["ci_high"], "p": c["p"], "p_holm": c["p_holm"]})
    s = json.loads((RUN / (n + "__sens.json")).read_text(encoding="utf-8"))
    for row in s["rows"]:
        d = row.get("vs_k_main") or row.get("federal_gain") or {}
        sens.append({"task": n, "k": row["k"], "method": row["method"], "cols_guest": row["guest"], "cols_host": row["host"],
                     "auroc": row["auroc"], "diff": d.get("diff"), "ci_low": d.get("ci_low"), "ci_high": d.get("ci_high"),
                     "p": d.get("p"), "vs": "k_main" if row.get("vs_k_main") else ("G_t" if row.get("federal_gain") else None)})
con, cvc, cvt, sens = (pd.DataFrame(x) for x in (con, cvc, cvt, sens))
tasks.to_csv(OUT / "tasks.csv", index=False)
con.to_csv(OUT / "contrasts.csv", index=False, float_format="%.5f")
cvc.to_csv(OUT / "cv_choices.csv", index=False, float_format="%.5f")
cvt.to_csv(OUT / "cv_contrasts.csv", index=False, float_format="%.5f")
sens.to_csv(OUT / "sensitivity.csv", index=False, float_format="%.5f")
R4 = cfg["round_auc"]
print("方法对比（测试 6 seed，两层 bootstrap；与 S5.M1 同口径）：")
print(con.round(R4).to_string(index=False))
print("组合选择：S5.M1 单一验证集 vs S5.M3 交叉验证：")
print(cvc.round(R4).to_string(index=False))
print(cvt.round(R4).to_string(index=False))
print("M3 预筛敏感性（final 视图，主分析配置）：")
print(sens.round(R4).to_string(index=False))
margin = float(m5["noninferiority_margin"])
full = sens[(sens["k"] == ML.ALL) & (sens["method"] == "F_opt_p")]
verdict = {r["task"]: bool(-r["ci_high"] > -margin) for _, r in full.iterrows()}
print("D-3 判定「预筛不损失精度」（k = %d 相对全部列：区间下界 > −%.3f）：" % (m5["k_main"], margin), verdict)

方法对比（测试 6 seed，两层 bootstrap；与 S5.M1 同口径）：
                  task        contrast    diff  ci_low  ci_high     p  p_holm  a_auroc
    orange_large_churn F_opt_p − F_def  0.0359  0.0229   0.0499 0.000   0.000   0.7545
    orange_large_churn     F_def − G_t  0.0053 -0.0108   0.0213 0.502   1.000   0.7186
    orange_large_churn   F_opt_p − G_t  0.0412  0.0290   0.0536 0.000   0.000   0.7545
    orange_large_churn  F_M4 − F_opt_p  0.0000  0.0000   0.0000 1.000   1.000   0.7545
    orange_large_churn  F_M1 − F_opt_p -0.0072 -0.0145  -0.0004 0.042   0.798   0.7472
    orange_large_churn  F_M6 − F_opt_p  0.0000 -0.0026   0.0028 0.960   1.000   0.7545
    orange_large_churn COMBO − F_opt_p -0.0072 -0.0151   0.0000 0.052   0.832   0.7472
   orange_large_upsell F_opt_p − F_def  0.0109  0.0039   0.0172 0.000   0.000   0.9146
   orange_large_upsell     F_def − G_t  0.0063 -0.0001   0.0135 0.066   1.000   0.9037
   orange_large_upsell   F_opt_p − G_t  0.0172  0.0119   0.0220 0.000   0.000   0.9146
 

## 4. 读法

**复现**：从原始分块重做流失任务的预筛，两方列清单与批量运行相同；优化版联邦测试 AUROC 重算与批量运行相同（0.754471）。

**方法在大版上的表现**（「优于 / 劣于」= 配对差 95% 区间整体在 0 以上 / 以下，与 S5.M1 同口径；Holm 见表）：

| 方法 | 流失 | 追加销售 | 偏好（测试正例 178） |
|---|---|---|---|
| I2 + I3（优化版 − 原版） | **+0.036** | **+0.011** | +0.025（Holm 0.86） |
| 优化版联邦 − 内地单边 | **+0.041** | **+0.017** | +0.018（Holm 0.15） |
| 原版联邦 − 内地单边 | +0.005（不显著） | +0.006（不显著） | −0.007（不显著） |
| M6 bagging | 0.000 | 0.000 | +0.005（区间下界 +0.00003） |
| M1 逐样本先验 | −0.007（区间上界 −0.0004） | −0.004（不显著） | 0.000 |
| M4 行采样 | 0.000 | 0.000 | −0.004（不显著） |

- 与小版和 14 个任务的结论一致：**不做 I2 + I3，联邦在三个任务上都看不出增益**；做了之后三个任务都优于内地单边。
- M6 只在偏好上刚好成立；M1 在流失上变差；M4 无增益。

**组合选择**：
- 单一验证集三个任务都选 M1：流失 −0.007（区间上界 +0.00002），其余不显著。
- 交叉验证 R1：流失 P+M6（0.000）、追加销售 M1+M6（−0.004，不显著）、偏好 P+M6（**+0.005 [+0.0002, +0.0097]**）。1 优于、0 劣于；与 14 个任务上的结论一致——交叉验证不再选出变差的组合。

**M3 预筛是否损失精度**（D-3，k = 300 相对全部列，区间下界 > −0.005）：
- 流失、追加销售：成立（全部列 − 300 列 = +0.0009 [−0.0029, +0.0046]、0.0000 [−0.0028, +0.0030]）。
- **偏好：不成立**——全部列 − 300 列 = +0.0050 [−0.0007, +0.0110]，不能排除 0.005 以上的损失。但 100 列反而比 300 列高 +0.0084 [+0.0015, +0.0158]，测试正例只有 178 个，不同 k 之间的差主要是噪声，不是单调的「列越多越好」。
- 每个 k 下优化版联邦都显著优于内地单边（+0.015 ~ +0.046）：联邦增益不依赖预筛的列数。
- 预筛把每方从约 6,650 列降到 300 列；SGB 的密文直方图开销随香港列数线性增长，这一步能把香港侧开销降约 20 倍（未在 SGB 上实测）。

**与小版并列**（行与标签不同，只并列、不配对）：大版优化版联邦 0.7545 / 0.9146 / 0.8687，小版 0.7305 / 0.8668 / 0.8496；联邦增益大版 +0.041 / +0.017 / +0.018，小版 +0.023 / −0.001 / +0.030。

**边界**：公开数据开发证据，不代表汇丰效果；测试集在训练集内构造；明文代理，未在真实 SGB 上复核；预筛依据的模型与 seed 固定（原版参数、seed 11）。技术侧自评，未经法务复核（W-001）。